# Clase 3 · Visión por computador II: arquitecturas avanzadas

**Curso:** Advanced Deep Learning (aprendizaje profundo avanzado)

Bienvenido a esta clase en directo. Vas a ver, con código funcionando, por qué las redes muy profundas necesitan una conexión residual y cómo reutilizar una red neuronal convolucional (CNN) ya entrenada en lugar de partir de cero.

## Objetivos de aprendizaje

Al final de esta clase, eres capaz de:

- Explicar por qué apilar demasiadas capas en una red profunda degrada su rendimiento
- Implementar un bloque residual en PyTorch y comprobar su paso hacia delante (forward pass)
- Cargar una CNN preentrenada con `torchvision` y usarla para clasificar una imagen
- Aplicar transferencia de aprendizaje (transfer learning): congelar capas y adaptar la última capa a un nuevo número de clases

## El problema de las redes muy profundas

Cuando apilas más capas en una red neuronal convolucional (CNN), esperas que el rendimiento mejore. Pero a partir de cierta profundidad ocurre lo contrario: el error de entrenamiento vuelve a subir. No es sobreajuste (overfitting): es un problema de optimización que se conoce como degradación.

La causa es el desvanecimiento del gradiente. Durante el entrenamiento, el gradiente se propaga hacia atrás capa a capa mediante la regla de la cadena. Si en cada capa el factor es menor que uno, el producto de muchos factores pequeños tiende a cero mucho antes de llegar a las primeras capas. Esas capas dejan entonces de aprender, aunque tengan capacidad suficiente para hacerlo.

Necesitas una forma de garantizar que el gradiente llegue con fuerza suficiente hasta las primeras capas, sin importar cuántas capas tenga la red. Esa es la motivación detrás de la conexión residual.

## La conexión residual

Imagina una ruta de senderismo con muchas etapas seguidas, todas cuesta arriba. Si pierdes algo de energía en cada etapa, al llegar a la etapa 50 puede que no te queden fuerzas para seguir. Eso es lo que le pasa al gradiente en una red muy profunda durante el entrenamiento.

Ahora imagina que, junto al camino principal, existe un atajo que baja directamente desde el inicio hasta cada refugio. No sustituye al camino principal, pero te garantiza siempre una vía directa.

Esa es la idea de ResNet (Residual Network, "red residual"): en vez de pedirle a un bloque de capas que aprenda una transformación completa H(x), le pides que aprenda solo el residuo F(x) = H(x) − x. La salida real del bloque es:

y = F(x) + x

Si la mejor opción para un bloque es dejar pasar la entrada sin cambios, la red solo necesita aprender a poner F(x) cerca de cero, mucho más fácil que aprender una identidad exacta con capas no lineales. Y durante la retropropagación, el gradiente dispone siempre de una vía directa a través de la suma, además de la vía que atraviesa las convoluciones.

Comprueba esta idea con un bloque residual construido a mano.

In [ ]:
import torch
import torch.nn as nn


class BloqueResidual(nn.Module):
    """Bloque residual: dos convoluciones 3x3 y un atajo (shortcut)
    que suma la entrada a la salida antes de la última activación."""

    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)
        self.relu = nn.ReLU(inplace=True)

        # Si el bloque cambia la resolución o el número de canales,
        # el atajo necesita una proyección 1x1 para poder sumarse a F(x)
        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida),
            )

    def forward(self, x):
        identidad = self.atajo(x)                 # atajo: y = F(x) + x
        salida = self.relu(self.bn1(self.conv1(x)))
        salida = self.bn2(self.conv2(salida))
        salida = salida + identidad                # aquí ocurre la conexión residual
        return self.relu(salida)


# Prueba rápida: lote de 2 imágenes, 3 canales, 32x32 píxeles
entrada_prueba = torch.randn(2, 3, 32, 32)
bloque = BloqueResidual(canales_entrada=3, canales_salida=16, stride=2)
salida_prueba = bloque(entrada_prueba)

print("Forma de entrada:", entrada_prueba.shape)
print("Forma de salida: ", salida_prueba.shape)

## Usa un modelo preentrenado

Entrenar una ResNet desde cero exige millones de imágenes y muchas horas de cómputo. No necesitas repetir ese trabajo: `torchvision` te da acceso directo a redes ya entrenadas sobre ImageNet, un banco de pruebas (benchmark) de más de un millón de imágenes repartidas en 1000 categorías.

Carga una ResNet-18 preentrenada, aplícala sobre una imagen real descargada de internet y observa sus cinco predicciones con mayor probabilidad (precisión top-5).

In [ ]:
import urllib.request

import torch
from PIL import Image
from torchvision import models, transforms

# Carga la ResNet-18 con los pesos entrenados sobre ImageNet
modelo = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
modelo.eval()  # desactiva el dropout y fija las estadísticas de BatchNorm

# Descarga una imagen de ejemplo y la lista de las 1000 clases de ImageNet
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
    "imagen_ejemplo.jpg",
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
    "clases_imagenet.txt",
)
with open("clases_imagenet.txt") as archivo:
    categorias = [linea.strip() for linea in archivo.readlines()]

imagen = Image.open("imagen_ejemplo.jpg").convert("RGB")
imagen

In [ ]:
preprocesar = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # medias de ImageNet
                          std=[0.229, 0.224, 0.225]),
])
entrada = preprocesar(imagen).unsqueeze(0)  # añade la dimensión de lote

with torch.no_grad():
    salida = modelo(entrada)
probabilidades = torch.nn.functional.softmax(salida[0], dim=0)

top5_prob, top5_idx = torch.topk(probabilidades, 5)
for probabilidad, indice in zip(top5_prob, top5_idx):
    print(f"{categorias[indice]}: {probabilidad.item() * 100:.1f} %")

## Transferencia de aprendizaje

La transferencia de aprendizaje (transfer learning) reutiliza lo que un modelo ya aprendió en una tarea de origen para acelerar el aprendizaje en una tarea de destino. La premisa es sencilla: muchas características visuales (bordes, texturas, formas) sirven para más de una tarea, así que no necesitas reaprenderlas cada vez.

El procedimiento habitual tiene dos pasos:

- **Congela** las capas preentrenadas: fija sus pesos para que no cambien durante el entrenamiento
- **Sustituye** la última capa por una nueva, adaptada al número de clases de tu problema

Así entrenas solo la capa nueva, con muchos menos datos y mucho menos tiempo que si partieras de cero.

In [ ]:
import torch.nn as nn
from torchvision import models

NUEVO_NUMERO_DE_CLASES = 10  # ajusta este valor a tu propio problema

modelo_transferencia = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congela todas las capas preentrenadas: no se actualizan durante el entrenamiento
for parametro in modelo_transferencia.parameters():
    parametro.requires_grad = False

# Sustituye la última capa lineal (fc) por una nueva, sin congelar
caracteristicas_entrada = modelo_transferencia.fc.in_features
modelo_transferencia.fc = nn.Linear(caracteristicas_entrada, NUEVO_NUMERO_DE_CLASES)

parametros_totales = sum(p.numel() for p in modelo_transferencia.parameters())
parametros_entrenables = sum(p.numel() for p in modelo_transferencia.parameters() if p.requires_grad)

print(f"Parámetros totales:     {parametros_totales:,}")
print(f"Parámetros entrenables: {parametros_entrenables:,}")

# Comprueba que la nueva capa produce la forma de salida esperada
entrada_ejemplo = torch.randn(1, 3, 224, 224)
modelo_transferencia.eval()
with torch.no_grad():
    salida_ejemplo = modelo_transferencia(entrada_ejemplo)
print("Forma de salida:", salida_ejemplo.shape)

## Resumen de la clase

- Apilar demasiadas capas sin control provoca el problema de la degradación: el gradiente se desvanece y la red deja de optimizarse bien
- La conexión residual reformula cada bloque para que aprenda un residuo, y = F(x) + x, lo que garantiza una vía directa para el gradiente
- Una CNN preentrenada, como ResNet-18 sobre ImageNet, te permite clasificar imágenes reales sin entrenar nada desde cero
- La transferencia de aprendizaje congela las capas preentrenadas y sustituye la última capa para adaptar el modelo a tu propio número de clases

**Abre ahora el notebook de práctica companion, `../notebooks_practica/clase03_practica_modelos_avanzados_vision.ipynb`, y pon en práctica lo que acabas de ver.**